In [1]:
#!uv add gitsource 

In [2]:
from dotenv import load_dotenv 
load_dotenv()

True

In [3]:
from gitsource import GithubRepositoryDataReader
from openai import OpenAI
from minsearch import Index

#### Data Ingestion  

In [4]:
reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub", 
    repo_name="llm-zoomcamp", 
    commit_id="8c1834d", 
    allowed_extensions={"md"}, 
    filename_filter=lambda path: "/lessons" in path, 
)

files = reader.read() 

In [5]:
documents = []
for file in files: 
    doc = file.parse() 
    documents.append(doc)

In [6]:
documents[0]

{'content': '# Introduction\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=rQYyFxf1FWw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn this module, we\'ll build a working Retrieval-Augmented\nGeneration (RAG) system from scratch, step by step.\n\nWe write everything in plain Python. We build a small search index by\nhand and call the LLM ourselves. I want you to see every piece first.\nThat way you know what a framework does for you before you reach for\none.\n\nPlaces where you can find me:\n\n- [My substack](https://alexeyondata.substack.com/)\n- [LinkedIn](https://www.linkedin.com/in/agrigorev/)\n- [X](https://x.com/Al_Grigor)\n\n## LLMs\n\nAn LLM (Large Language Model) is a neural network trained on massive\namounts of text. Given a prompt, it generates a continuation - a\nplausible next piece of text.\n\nThink of your phone. When you type "how are" in WhatsApp, it suggests\n"you" as the next word. "How are you" is the most common continuation.\nYour phone uses a simp

#### Q1 Solution - How many lesson pages 

In [7]:
print(f"The number of lesson pages is {len(documents)}")

The number of lesson pages is 72


#### Q2 Solution - Indexing and Searching 

In [8]:
index = Index(
    text_fields=["content"], 
    keyword_fields=["filename"]
)
index.fit(documents)

q2_query = "How does the agentic loop keep calling the model until it stops?"
search_results = index.search(
    q2_query,
)

print(f"The filename of the first result is {search_results[0]['filename']}")

The filename of the first result is 01-agentic-rag/lessons/14-agentic-loop.md


#### Q3 Solution - RAG  

In [9]:
# To import the RAGBase module, I add the project root to sys.path 

from pathlib import Path
import sys

root = Path.cwd().resolve().parents[0] 
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

Let's import the RAGBase module, and redefine the `search`, `build_context`, `llm`, and `rag` functions for the current assignment.  

In [10]:
from notes_lesson_1_rag.rag_helper import RAGBase

def homework_search(self, query, num_results=5): 
    return self.index.search(
        query, 
        num_results=num_results
    ) 


def homework_build_context(self, search_results): 
        lines = []

        for doc in search_results:
            lines.append(doc["filename"])
            lines.append(doc["content"])
            lines.append("")

        return "\n".join(lines).strip()


def homework_llm(self, prompt, return_raw=False):
    input_messages = [
        {"role": "developer", "content": self.instructions},
        {"role": "user", "content": prompt}
    ]

    response = self.llm_client.responses.create(
        model=self.model,
        input=input_messages
    )

    return response.output_text if not return_raw else response
    

def homework_rag(self, query, return_raw=False):
    search_results = self.search(query)
    prompt = self.build_prompt(query, search_results)
    answer = self.llm(prompt, return_raw)
    return answer


RAGBase.search = homework_search
RAGBase.build_context = homework_build_context
RAGBase.llm = homework_llm
RAGBase.rag = homework_rag

In [11]:
openai_client = OpenAI()
rag_assistant = RAGBase(
    index=index, 
    llm_client=openai_client,
    course=None, 
    model="gpt-5.4-mini"
)

Test that the `search` and `build_context` functions work on the assignment documents: 

In [12]:
search_results = rag_assistant.search("How does the agentic loop keep calling the model until it stops?")

assert search_results[0]["filename"] == "01-agentic-rag/lessons/14-agentic-loop.md"
assert rag_assistant.build_context(search_results)[:60] == "01-agentic-rag/lessons/14-agentic-loop.md\n# The Agentic Loop"

The answer to the exercise: 

In [13]:
q3_query = "How does the agentic loop keep calling the model until it stops?" 

raw_response = rag_assistant.rag(q3_query, return_raw=True) 
n_input_tokens = raw_response.usage.input_tokens

print(f"The number of sent input tokens is {n_input_tokens} (~7000)")

The number of sent input tokens is 7111 (~7000)


#### Q4 Solution - Chunking 

In [14]:
from gitsource import chunk_documents

chunks = chunk_documents(documents, size=2_000, step=1_000)

In [15]:
# Inspect chunks
chunks[:5]

[{'start': 0,
  'content': '# Introduction\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=rQYyFxf1FWw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn this module, we\'ll build a working Retrieval-Augmented\nGeneration (RAG) system from scratch, step by step.\n\nWe write everything in plain Python. We build a small search index by\nhand and call the LLM ourselves. I want you to see every piece first.\nThat way you know what a framework does for you before you reach for\none.\n\nPlaces where you can find me:\n\n- [My substack](https://alexeyondata.substack.com/)\n- [LinkedIn](https://www.linkedin.com/in/agrigorev/)\n- [X](https://x.com/Al_Grigor)\n\n## LLMs\n\nAn LLM (Large Language Model) is a neural network trained on massive\namounts of text. Given a prompt, it generates a continuation - a\nplausible next piece of text.\n\nThink of your phone. When you type "how are" in WhatsApp, it suggests\n"you" as the next word. "How are you" is the most common continuation.\nYour ph

In [16]:
print(f"The number of obtained chunks is {len(chunks)}")

The number of obtained chunks is 295


##### Q5 Solution - RAG with Chunking 

In [17]:
chunk_index = Index(
    text_fields=["content"], 
    keyword_fields=["filename"]
)
chunk_index.fit(chunks)

openai_client = OpenAI()
chunks_rag_assistant = RAGBase(
    index=chunk_index, 
    llm_client=openai_client,
    course=None, 
    model="gpt-5.4-mini"
)

q5_query = q3_query
raw_response = chunks_rag_assistant.rag(q5_query, return_raw=True) 
n_input_tokens_chunks = raw_response.usage.input_tokens

In [18]:
print(f"The number of sent input tokens after chunking is {n_input_tokens_chunks}, with a reduction of ~{n_input_tokens/n_input_tokens_chunks:.2f}x")

The number of sent input tokens after chunking is 2294, with a reduction of ~3.10x


#### Q6 Solution - Turning it into an agent

In [19]:
from toyaikit.llm import OpenAIClient 
from toyaikit.tools import Tools 
from toyaikit.chat.runners import OpenAIResponsesRunner

In [20]:
# Register tool from Python function 
def search(query: str) -> dict[str, str]:
    """
    Search the lesson pages for entries matching the given query.
    """
    return chunk_index.search(
        query,
        num_results=5,
    )

agent_tools = Tools()
agent_tools.add_tool(search)

In [21]:
# Check that the tool is registered: 
agent_tools.get_tools()

[{'type': 'function',
  'name': 'search',
  'description': 'Search the lesson pages for entries matching the given query.',
  'parameters': {'type': 'object',
   'properties': {'query': {'type': 'string',
     'description': 'query parameter'}},
   'required': ['query'],
   'additionalProperties': False}}]

In [22]:
instructions = """
You're a course teaching assistant. Answer the student's question using the search tool. Make multiple searches with different keywords before answering.
""" 

q6_query = "How does the agentic loop work, and how is it different from plain RAG?"

In [23]:
# chat_interface = IPythonChatInterface() 
runner = OpenAIResponsesRunner(
    tools=agent_tools,
    developer_prompt=instructions,
    # chat_interface=chat_interface,
    llm_client=OpenAIClient(model="gpt-5.4-mini")
)

result = runner.loop(
    prompt=q6_query, 
)

In [24]:
n_function_calls = 0 

for message in result.all_messages: 
    if type(message) is not dict and message.type == "function_call": 
        n_function_calls += 1 

print(f"The number of function calls operated by the agent was {n_function_calls}.")

The number of function calls operated by the agent was 3.
